In [ ]:
import numpy as np
import os
from tqdm import tqdm
from dotenv import load_dotenv
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter

from rfml_uav.drone_rf.consts import BUI
from rfml_uav.drone_rf.utils import get_class_label, get_input_dim

load_dotenv()
np.random.seed(1)
os.environ['OMP_NUM_THREADS'] = '1'

In [16]:
feature = "PSD" # select a feature to balance a corresponding dataset
load_path = os.path.join(os.getenv("DATA_PATH"), feature)
save_path = os.path.join(os.getenv("DATA_PATH"), "balanced")
os.makedirs(save_path, exist_ok=True)

In [17]:
input_dim = get_input_dim(feature)
X = np.empty((0, input_dim))
y = np.array([])

for bui in tqdm(BUI):
    file_path = os.path.join(load_path, f"{bui}.csv")
    x = np.loadtxt(file_path, delimiter=",")
    X = np.vstack((X, x))
    
    labels = np.empty(len(x))
    labels.fill(get_class_label(bui))
    y = np.concat((y, labels))

100%|██████████| 10/10 [08:43<00:00, 52.36s/it]


In [18]:
# Only undersample the majority class (background noise)
rus = RandomUnderSampler(random_state=42, sampling_strategy="majority")
X_res, y_res = rus.fit_resample(X, y)
print('Resampled dataset shape %s' % Counter(y_res))

Resampled dataset shape Counter({np.float64(2.0): 130285, np.float64(3.0): 113588, np.float64(0.0): 102500, np.float64(1.0): 102500})


In [19]:
np.savez_compressed(os.path.join(save_path, f'{feature}.npz'), X=X_res, y=y_res)